# Online Retail II — Cleaning, RFM & Cohort Retention

**Business context.** Two years (Dec 2009 – Dec 2011) of transactions for a UK-based online gift retailer. This notebook prepares the data for an executive dashboard answering: *where does revenue come from, which customers drive value, and do they come back?*

**What this notebook produces** (all saved to `../data/processed/` for Tableau):
- `clean_sales_all.csv` — all valid sales (incl. unidentified customers) → commercial KPIs
- `clean_sales_identified.csv` — identified customers only → RFM / cohort
- `customer_metrics.csv`, `segment_summary.csv` — RFM scores & segments
- `cohort_retention.csv` — monthly retention matrix (feeds the heat map)
- `returns.csv`, `repeat_vs_onetime.csv`

**Cleaning decisions (log).** Drop 34k exact duplicate rows; split cancellations/returns (invoices starting `C` or negative quantity) into a separate table; keep two sales layers so commercial revenue isn't understated by the ~15% of sales from unidentified customers, while RFM stays on identified customers only; drop zero/negative-price rows.

## 1. Load & inspect

In [ ]:
import pandas as pd
import numpy as np

raw = pd.read_csv("../data/raw/online_retail_II.csv", dtype={"Invoice": str, "StockCode": str})
raw["InvoiceDate"] = pd.to_datetime(raw["InvoiceDate"])

print("Shape:", raw.shape)
display(raw.head())
raw.isnull().sum()

## 2. Clean & shape

Remove exact duplicates, compute `Revenue`, separate returns, and build the two sales layers.

In [ ]:
df = raw.drop_duplicates().copy()
print(f"Dropped {len(raw) - len(df):,} exact-duplicate rows")

df["Revenue"] = df["Quantity"] * df["Price"]

# returns / cancellations: invoices starting 'C' or negative quantity
is_return = df["Invoice"].str.startswith("C", na=False) | (df["Quantity"] < 0)
returns_df = df[is_return].copy()

# layer A: all valid sales (incl. unidentified customers) -> commercial KPIs
sales_all = df[(~is_return) & (df["Price"] > 0)].copy()
sales_all["YearMonth"] = sales_all["InvoiceDate"].dt.to_period("M").astype(str)
sales_all["Country"] = sales_all["Country"].replace("EIRE", "Ireland")

# layer B: identified customers only -> RFM / cohort
sales_identified = sales_all.dropna(subset=["Customer ID"]).copy()
sales_identified["Customer ID"] = sales_identified["Customer ID"].astype("int64")

print(f"Returns/cancellations : {len(returns_df):,}")
print(f"Sales (all)           : {len(sales_all):,} | total revenue GBP {sales_all['Revenue'].sum():,.0f}")
print(f"Sales (identified)    : {len(sales_identified):,} | revenue GBP {sales_identified['Revenue'].sum():,.0f}")

## 3. Commercial snapshot

Headline commercial views run on **all sales** (so revenue is complete).

In [ ]:
# revenue by country (top 10)
country = (sales_all.groupby("Country")["Revenue"].sum()
           .sort_values(ascending=False).round(2).reset_index())

# top products, excluding non-product lines (postage, manual adjustments, fees)
non_products = ["Manual", "POSTAGE", "DOTCOM POSTAGE", "Adjust bad debt", "AMAZON FEE", "Bank Charges"]
products = (sales_all[~sales_all["Description"].isin(non_products)]
            .groupby("Description")["Revenue"].sum()
            .sort_values(ascending=False).round(2).reset_index())

display(country.head(10))
display(products.head(10))

## 4. RFM scoring & segmentation

Score each identified customer on Recency, Frequency, Monetary (quintiles), then map to segments.

In [ ]:
def segment_customer(row):
    r, f, m = int(row["R_Score"]), int(row["F_Score"]), int(row["M_Score"])
    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    elif r >= 3 and f >= 3 and m >= 3:
        return "Loyal Customers"
    elif r >= 4 and f <= 2:
        return "Potential Loyalists"
    elif r <= 2 and m >= 4:
        return "At Risk"
    else:
        return "Others"

snapshot = sales_identified["InvoiceDate"].max()
cm = (sales_identified.groupby("Customer ID")
      .agg(TotalOrders=("Invoice", "nunique"),
           TotalRevenue=("Revenue", "sum"),
           LastPurchaseDate=("InvoiceDate", "max"))
      .reset_index().rename(columns={"Customer ID": "CustomerID"}))
cm["Recency"] = (snapshot - cm["LastPurchaseDate"]).dt.days
cm["R_Score"] = pd.qcut(cm["Recency"], 5, labels=[5, 4, 3, 2, 1]).astype(int)
cm["F_Score"] = pd.qcut(cm["TotalOrders"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
cm["M_Score"] = pd.qcut(cm["TotalRevenue"], 5, labels=[1, 2, 3, 4, 5]).astype(int)
cm["RFM_Score"] = cm[["R_Score", "F_Score", "M_Score"]].astype(str).agg("".join, axis=1)
cm["Segment"] = cm.apply(segment_customer, axis=1)

seg = (cm.groupby("Segment")
       .agg(CustomerCount=("CustomerID", "count"),
            TotalRevenue=("TotalRevenue", "sum"),
            AvgOrders=("TotalOrders", "mean")).reset_index())
seg["RevenuePct"]  = (seg["TotalRevenue"] / seg["TotalRevenue"].sum() * 100).round(1)
seg["CustomerPct"] = (seg["CustomerCount"] / seg["CustomerCount"].sum() * 100).round(1)
seg = seg.sort_values("TotalRevenue", ascending=False)
display(seg)

## 5. Cohort retention — feeds the heat map

Assign each customer an acquisition month (first purchase), then measure what share of that cohort is still active in each subsequent month. The output is a long-format matrix Tableau plots as a heat map (rows = `CohortMonth`, columns = `CohortIndex`, color = `RetentionPct`).

In [ ]:
ci = sales_identified.copy()
ci["InvoiceMonth"] = ci["InvoiceDate"].dt.to_period("M")
ci["CohortMonth"]  = ci.groupby("Customer ID")["InvoiceMonth"].transform("min")
ci["CohortIndex"]  = ((ci["InvoiceMonth"].dt.year - ci["CohortMonth"].dt.year) * 12
                      + (ci["InvoiceMonth"].dt.month - ci["CohortMonth"].dt.month))

g = (ci.groupby(["CohortMonth", "CohortIndex"])["Customer ID"]
       .nunique().reset_index(name="ActiveCustomers"))
sizes = (g[g["CohortIndex"] == 0][["CohortMonth", "ActiveCustomers"]]
         .rename(columns={"ActiveCustomers": "CohortSize"}))
cohort = g.merge(sizes, on="CohortMonth")
cohort["RetentionPct"] = (cohort["ActiveCustomers"] / cohort["CohortSize"] * 100).round(1)
cohort["CohortMonth"] = cohort["CohortMonth"].astype(str)
cohort = cohort[["CohortMonth", "CohortIndex", "CohortSize", "ActiveCustomers", "RetentionPct"]]

# repeat vs one-time customers (story stat)
orders = sales_identified.groupby("Customer ID")["Invoice"].nunique()
rev    = sales_identified.groupby("Customer ID")["Revenue"].sum()
repeat = orders > 1
split = pd.DataFrame({"Group": ["Repeat customers", "One-time customers"],
                      "Customers": [int(repeat.sum()), int((~repeat).sum())],
                      "Revenue": [round(rev[repeat].sum(), 2), round(rev[~repeat].sum(), 2)]})
split["RevenuePct"]  = (split["Revenue"] / split["Revenue"].sum() * 100).round(1)
split["CustomerPct"] = (split["Customers"] / split["Customers"].sum() * 100).round(1)

display(cohort.head(13))
display(split)

## 6. Export for Tableau

In [ ]:
out = "../data/processed/"
sales_all.to_csv(out + "clean_sales_all.csv", index=False)
sales_identified.to_csv(out + "clean_sales_identified.csv", index=False)
returns_df.to_csv(out + "returns.csv", index=False)
cm.to_csv(out + "customer_metrics.csv", index=False)
seg.to_csv(out + "segment_summary.csv", index=False)
cohort.to_csv(out + "cohort_retention.csv", index=False)
split.to_csv(out + "repeat_vs_onetime.csv", index=False)
print("Exported 7 files to data/processed/")